# Model Selection and Evaluation - Part 1

When building supervised machine learning models, we need to solve two
problems:

1. **Model selection** - Finding the model that does as well as possible
on our learning task.

2. **Model evaluation** - Predicting **generalization error**, or the expected performance of our
model on unseen data.

Both are critical.  Without 1. we can't have an effective model and
without 2. we can't *know* if we have an effective model.

Once we have picked a particular machine learning algorithm, model
selection comes down to the problem of **hyperparameter** tuning.
Hyperparameters are parameters of our learning models that need to be
selected before the model can be learned.  Examples include the
maximum number of leaves in a decision tree or the number of hidden
units in a neural network classifier.

**WARNING:**  In the exercises below we will try several *BAD* approaches to model selection and evaluation.  These examples are not meant to illustrate the correct way of doing things, they are meant to show the consequences of doing things incorrectly. 


## Exercise 1 - Naive Model Selection

For now, let's focus entirely on model selection and disregard model
evaluation.  The following cell will load a data set and use a
decision tree regressor to fit a decision tree to the data. Try adjusting the `max_leaf_nodes` hyperparameter in order to minimize the MSE on the training set.

In [ ]:
%matplotlib qt
# %matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import datasource

from sklearn.tree import DecisionTreeRegressor

# Grab our training data
source = datasource.DataSource()
X, y = source.gen_data(100, seed=100)

# Build a decision tree regressor
tree = DecisionTreeRegressor(max_leaf_nodes=3)
tree.fit(X, y)

# Evaluate the MSE of our decision tree on the training set
y_predict = tree.predict(X)
mse = np.sum((y - y_predict)**2) / y.size
print("MSE: {:.4f}".format(mse))

# Plot the fit.
plt.plot(X, y, '*')
x_plt = np.linspace(0, 1, 400).reshape(400, 1)
plt.plot(x_plt, tree.predict(x_plt))
plt.show()


In [ ]:
# Systematically search max_leaf_nodes using the TRAINING data.
leaf_values = range(2, X.shape[0] + 1)
train_mses = []

for leaves in leaf_values:
    candidate = DecisionTreeRegressor(max_leaf_nodes=leaves)
    candidate.fit(X, y)
    pred = candidate.predict(X)
    train_mses.append(np.mean((y - pred) ** 2))

best_train_index = int(np.argmin(train_mses))
best_train_leaves = list(leaf_values)[best_train_index]
best_train_mse = train_mses[best_train_index]
print(f'Best training max_leaf_nodes: {best_train_leaves}')
print(f'Lowest training MSE: {best_train_mse:.6f}')

plt.figure()
plt.plot(list(leaf_values), train_mses)
plt.xlabel('max leaf nodes')
plt.ylabel('Training MSE')
plt.title('Naive Model Selection on Training Data')
plt.show()


### Questions

* What value of the hyperparameter resulted in the lowest MSE?
* Do you think that this MSE reflects how well this model will do on unseen data?  Why or why not?

### Answers

* The lowest training-set MSE is obtained by using the most flexible tree considered in the search. A tree with enough leaf nodes can fit the training observations extremely closely, so the training MSE can approach zero. The completed search cell below finds and prints the exact best value rather than assuming a particular value.
* No. Training MSE is not a reliable estimate of performance on unseen data because the hyperparameter was selected using the same observations used to fit the model. Increasing tree complexity can reduce training error while increasing overfitting and therefore worsening generalization.

In the exercise above, you were able to tune the hyperparameters so as to *perfectly* fit the training data.  Now let's see what happens when we use this model on some new data drawn from the same underlying distribution:

In [ ]:
X_new, y_new = source.gen_data(1000, seed=200)
y_new_predict = tree.predict(X_new)
mse = np.sum((y_new - y_new_predict)**2) / y_new.size
print("MSE: {:.4f}".format(mse))

## Exercise 2 - Using a Test Set for Hyperparameter Tuning and Evaluation

In the exercise above, you were able to perfectly fit a training data set, but that didn't tell you anything about how well your model would perform on unseen data.  We might address this by splitting our limited data into a training set and a test set.  This is illustrated in the cell below.

In [ ]:
# Systematically tune max_leaf_nodes on the held-out TESTING data.
test_leaf_values = range(2, X_train.shape[0] + 1)
test_mses = []

for leaves in test_leaf_values:
    candidate = DecisionTreeRegressor(max_leaf_nodes=leaves)
    candidate.fit(X_train, y_train)
    pred = candidate.predict(X_test)
    test_mses.append(np.mean((y_test - pred) ** 2))

best_test_index = int(np.argmin(test_mses))
best_test_leaves = list(test_leaf_values)[best_test_index]
best_test_mse = test_mses[best_test_index]
print(f'Best testing max_leaf_nodes: {best_test_leaves}')
print(f'Lowest testing MSE: {best_test_mse:.6f}')

plt.figure()
plt.plot(list(test_leaf_values), test_mses)
plt.xlabel('max leaf nodes')
plt.ylabel('Testing MSE')
plt.title('Hyperparameter Selection Using the Test Set')
plt.show()


### Answers

* The completed search below evaluates every candidate number of leaves on the held-out test set and stores the best setting in `best_test_leaves`. The exact value and MSE are printed.
* This test-set MSE is useful for selecting a hyperparameter, but it should **not** be treated as an unbiased final estimate of generalization error because the test set was used to choose the hyperparameter.

# Use the best hyperparameter found from the test-set search above.
tree = DecisionTreeRegressor(max_leaf_nodes=best_test_leaves)
tree.fit(X_train, y_train)

# Evaluate on new, unobserved data.
X_new, y_new = source.gen_data(1000, seed=200)
y_new_predict = tree.predict(X_new)
mse = np.mean((y_new - y_new_predict) ** 2)
print(f'Generalization MSE on additional unseen data: {mse:.4f}')


In [ ]:
tree = DecisionTreeRegressor(max_leaf_nodes=????) # Put your best hyperparameter here!
tree.fit(X_train, y_train)

# Let's see how we do on unobserved data... 
X_new, y_new = source.gen_data(1000, seed=200)
y_new_predict = tree.predict(X_new)
mse = np.sum((y_new - y_new_predict)**2) / y_new.size
print("MSE: {:.4f}".format(mse))

### Answers

* Relative to Exercise 1, the train/test split is beneficial for **model selection** because it prevents us from choosing a model solely because it fits the training observations. However, selecting directly on the test set still introduces information leakage: the test set influences the chosen hyperparameter.
* The split is only partially beneficial for **model evaluation**. It is better than reporting training error, but the same test set was used for tuning, so its error is optimistically biased and is not a clean estimate of generalization error.
* The main problem is that the test set is serving two roles: validation for hyperparameter selection and evaluation of the final model. The additional unseen-data experiment demonstrates how different the test-set estimate can be from the actual performance on new data. A separate test set or cross-validation should be used for a more reliable evaluation.

### Click [here](model_selection_2.ipynb) to open the next page of exercises...